In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# ---------------------------------------------------------
# 1. DATA LOADING & INITIAL CLEANING
# ---------------------------------------------------------
# Target file generated from your Sampling and Elastic Net steps
df = pd.read_csv('FOR_ML.csv')
df = df.dropna()

print("--- Step 1: Data Acquisition Complete ---")
print(f"Total patient records processed (Sampled Population): {len(df):,}")

# ---------------------------------------------------------
# 2. FEATURE SELECTION & ENCODING VERIFICATION
# ---------------------------------------------------------
# Target is 'los'. 'id' is retained for reference but excluded from training.
y = df['los']
X = df.drop(columns=['los', 'id'], errors='ignore')

# Ensuring clinical categorical groups remain numerical
categorical_cols = ['gender', 'age', 'ward', 'discipline', 'dis_shift', 'holiday_3', 'holiday_4', 'holiday_5']

for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype(int)

print(f"--- Step 2: Feature Engineering Complete ({X.shape[1]} Features Verified) ---")

# ---------------------------------------------------------
# 3. DATA PARTITIONING & MODEL TRAINING
# ---------------------------------------------------------
# Using your preferred 80/20 split and seed 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42
)

# Hyperparameters
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.300000012,
    subsample=1,
    colsample_bytree=1,
    colsample_bylevel=1,
    reg_alpha=0,
    reg_lambda=1,
    gamma=0,
    objective='reg:squarederror',
    random_state=0,
    n_jobs=-1,
    tree_method='auto'
)

print("Training the XGBoost model...")
model.fit(X_train, y_train)

print("--- Step 3: Model Development Complete ---")

# ---------------------------------------------------------
# 4. EVALUATION (TRAIN VS TEST)
# ---------------------------------------------------------
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("\n--- Step 4 & 5: Dissertation Results ---")

# Training Set Metrics
print(f"TRAIN - Mean Absolute Error (MAE): {mean_absolute_error(y_train, y_train_pred):.4f} days")
print(f"TRAIN - Root Mean Squared Error:    {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f} days")
print(f"TRAIN - R-squared (R2) Score:       {r2_score(y_train, y_train_pred):.4f}")

print("-" * 50)

# Testing Set Metrics
print(f"TEST  - Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_test_pred):.4f} days")
print(f"TEST  - Root Mean Squared Error:    {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f} days")
print(f"TEST  - R-squared (R2) Score:       {r2_score(y_test, y_test_pred):.4f}")

# ---------------------------------------------------------
# 5. FEATURE IMPORTANCE
# ---------------------------------------------------------
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop 15 Clinical Predictors for LOS:")
print(importance_df.head(15))